<a href="https://www.kaggle.com/code/abhishekgodara/translation-akkadian-english-score-34-2?scriptVersionId=292109457" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import re
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# =============================================================================
# CONFIGURATION
# =============================================================================
TEST_DATA_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"

MODEL1_PATH = "/kaggle/input/byt5-base-big-data2"
MODEL2_PATH = "/kaggle/input/byt5-akkadian-model"
MODEL3_PATH = "/kaggle/input/train-gap-all-2/byt5-base-akkadian_gap_setence2"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PREFIX = "translate Akkadian to English: "
MAX_SOURCE_LEN = 512
BATCH_SIZE = 8

# Generation parameters - KEEP YOUR WORKING VALUES
NUM_BEAMS = 9
MAX_NEW_TOKENS = 512
LENGTH_PENALTY = 1.10
EARLY_STOPPING = True

# =============================================================================
# INPUT PREPROCESSING - Your original (it works!)
# =============================================================================
def preprocess_input(text):
    """Normalize gaps in INPUT (transliteration) text."""
    if pd.isna(text):
        return ""
    text = str(text)
    
    # Multi-dot patterns → <big_gap>
    text = re.sub(r'\.{3}(?:\s+\.{3})+', '<big_gap>', text)
    text = re.sub(r'\.\.\.\.\.\.', '<big_gap>', text)
    text = re.sub(r'\.\.\.', '<big_gap>', text)
    text = re.sub(r'……', '<big_gap>', text)
    text = re.sub(r'…', '<big_gap>', text)
    
    # x patterns → <gap>
    text = re.sub(r'xx+', '<gap>', text)
    text = re.sub(r' x ', ' <gap> ', text)
    
    return text

# =============================================================================
# OUTPUT POST-PROCESSING - Your original (it works!)
# =============================================================================
def postprocess_output(text):
    """
    Post-processing with proven optimizations:
    1. ḫ→h conversion
    2. Subscript number normalization
    3. Gap normalization
    4. Character denoising
    """
    if not isinstance(text, str):
        return str(text) if text else "broken text"
    
    if not text.strip():
        return "broken text"
    
    # ḫ/Ḫ → h/H conversion
    text = text.replace('ḫ', 'h')
    text = text.replace('Ḫ', 'H')
    
    # Subscript number normalization
    subscript_map = {
        '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
        '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9',
    }
    for sub, normal in subscript_map.items():
        text = text.replace(sub, normal)
    
    # Convert gap patterns in output to proper tags
    text = re.sub(r'\[x\]', '<gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'\(x\)', '<gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'\bx\b', '<gap>', text)
    text = re.sub(r'\.\.\.+', '<big_gap>', text)
    text = re.sub(r'…', '<big_gap>', text)
    text = re.sub(r'\[\.+\]', '<big_gap>', text)
    
    # Gap consolidation (same-type only!)
    text = re.sub(r'<gap>\s*<gap>', '<big_gap>', text)
    text = re.sub(r'<big_gap>\s*<big_gap>', '<big_gap>', text)
    
    # Remove scribal annotations
    scribal_patterns = [
        r'\(fem\.\s*plur\.\)', r'\(fem\.\)', r'\(plur\.\)', 
        r'\(pl\.\)', r'\(sing\.\)', r'\(singular\)', r'\(plural\)',
        r'\(\?\)', r'\(!\)',
    ]
    for pattern in scribal_patterns:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # Character denoising
    text = text.replace('<gap>', '\x00GAP\x00')
    text = text.replace('<big_gap>', '\x00BIGGAP\x00')
    
    forbidden_chars = [
        '!', '?', '(', ')', '"', '"', '"', ';', '—', '–',
        '<', '>', '⌈', '⌊', '⌋', '[', ']', '+', 'ʾ', '/',
    ]
    for char in forbidden_chars:
        text = text.replace(char, '')
    
    text = text.replace('\x00GAP\x00', '<gap>')
    text = text.replace('\x00BIGGAP\x00', '<big_gap>')
    
    # Fraction normalization
    text = re.sub(r'(\d+)\.5\b', r'\1 ½', text)
    text = re.sub(r'(\d+)\.33+\d*\b', r'\1 ⅓', text)
    text = re.sub(r'(\d+)\.66+\d*\b', r'\1 ⅔', text)
    text = re.sub(r'(\d+)\.25\b', r'\1 ¼', text)
    text = re.sub(r'(\d+)\.75\b', r'\1 ¾', text)
    
    text = re.sub(r'\b0\.5\b', '½', text)
    text = re.sub(r'\b0\.33+\d*\b', '⅓', text)
    text = re.sub(r'\b0\.66+\d*\b', '⅔', text)
    text = re.sub(r'\b0\.25\b', '¼', text)
    text = re.sub(r'\b0\.75\b', '¾', text)
    
    # After / removal, "1/2" becomes "1 2"
    text = text.replace(' 1 2', ' ½')
    text = text.replace(' 1 3', ' ⅓')
    text = text.replace(' 2 3', ' ⅔')
    text = text.replace(' 1 4', ' ¼')
    text = text.replace(' 3 4', ' ¾')
    
    # Final cleanup
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    text = re.sub(r'^-\s*', '', text)
    text = re.sub(r'\s*-$', '', text)
    
    return text if text else "broken text"

# =============================================================================
# MODEL LOADING - Your original ensemble (it works!)
# =============================================================================
perf1, perf2, perf3 = 0.98, 1.00, 0.40
total = perf1 + perf2 + perf3
w1, w2, w3 = perf1/total, perf2/total, perf3/total
print(f"Ensemble Weights: M1={w1:.4f}, M2={w2:.4f}, M3={w3:.4f}")

print("Loading models for ensemble...")
m1 = AutoModelForSeq2SeqLM.from_pretrained(MODEL1_PATH)
m2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL2_PATH)
m3 = AutoModelForSeq2SeqLM.from_pretrained(MODEL3_PATH)

sd1, sd2, sd3 = m1.state_dict(), m2.state_dict(), m3.state_dict()

print("Averaging model weights...")
final_sd = sd2.copy()
for k in final_sd:
    if k in sd1 and k in sd3:
        final_sd[k] = w1 * sd1[k] + w2 * sd2[k] + w3 * sd3[k]
    elif k in sd1:
        final_sd[k] = w1 * sd1[k] + (w2 + w3) * sd2[k]
    elif k in sd3:
        final_sd[k] = w3 * sd3[k] + (w1 + w2) * sd2[k]

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL2_PATH)
model.load_state_dict(final_sd)
model.to(DEVICE).eval()
model.float()

tokenizer = AutoTokenizer.from_pretrained(MODEL2_PATH)

# =============================================================================
# DATASET AND DATALOADER - Your original (it works!)
# =============================================================================
test_df = pd.read_csv(TEST_DATA_PATH)
test_df["transliteration"] = test_df["transliteration"].apply(preprocess_input)

class InferenceDataset(Dataset):
    """Dataset for inference - loads test data with preprocessing."""
    def __init__(self, df):
        self.ids = df["id"].tolist()
        self.texts = [PREFIX + t for t in df["transliteration"].astype(str).tolist()]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.ids[idx], self.texts[idx]

def collate_fn(batch):
    """Collate function - tokenizes batch of texts."""
    ids, texts = zip(*batch)
    enc = tokenizer(
        list(texts),
        max_length=MAX_SOURCE_LEN,
        truncation=True,
        padding=True,
        return_tensors="pt"
    )
    return list(ids), enc["input_ids"], enc["attention_mask"]

loader = DataLoader(
    InferenceDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=collate_fn
)

# =============================================================================
# INFERENCE - Your original (it works!)
# =============================================================================
all_ids, all_pred = [], []
torch.set_grad_enabled(False)

print("Starting inference...")
print(f"  NUM_BEAMS: {NUM_BEAMS}")
print(f"  MAX_NEW_TOKENS: {MAX_NEW_TOKENS}")
print(f"  LENGTH_PENALTY: {LENGTH_PENALTY}")

with torch.inference_mode():
    for batch_idx, (ids, input_ids, attention_mask) in enumerate(loader):
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        # Generate translations using beam search
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_NEW_TOKENS,
            length_penalty=LENGTH_PENALTY,
            early_stopping=EARLY_STOPPING,
        )

        # Decode and post-process outputs
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        decoded = [postprocess_output(d) for d in decoded]

        all_ids.extend(ids)
        all_pred.extend(decoded)
        
        if (batch_idx + 1) % 10 == 0:
            print(f"Processed batch {batch_idx + 1}/{len(loader)}")

# =============================================================================
# OUTPUT - Your original (it works!)
# =============================================================================
submission = pd.DataFrame({"id": all_ids, "translation": all_pred})
submission.to_csv("submission.csv", index=False)
print(f"\n✓ Saved submission.csv with {len(submission)} rows")
print("\nSample predictions:")
print(submission.head(10).to_string(index=False))

Loading models...


2026-01-15 21:58:25.264774: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768514305.461121      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768514305.517310      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768514305.991630      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768514305.991669      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768514305.991672      55 computation_placer.cc:177] computation placer alr

Weights → w1=0.412, w2=0.420, w3=0.168
✅ submission.csv saved
   id                                        translation
0   0  Thus Kanesh colony, say to the <big_gap> of ou...
1   1  In the tablet of the City you wrote to me in t...
2   2  Just as you hear our letter, he has given eith...
3   3  I sent our certified tablets to every single d...
